# 02 — Monthly update and rebalance

**What this does:** refreshes provisional live prices, rebuilds the research panel, creates the current target portfolio and trade list, presents a current portfolio summary, and runs the required human reviews.

**Run first:** notebook 01 must have completed successfully at least once. `data/current_portfolio.csv` must contain the agreed current holdings and weights.

**Produces:** an updated `data/live_prices.parquet`, updated configured panel, and date-stamped `data/target_portfolio_YYYYMMDD.csv` and `data/trade_list_YYYYMMDD.csv`. It deliberately does **not** overwrite the current-portfolio file.

**Time:** usually 5–20 minutes. Run this notebook one cell at a time. The quarantine and trade-list sections are mandatory human checkpoints, not informational decoration.

## Step 1 — Load the project and confirm operating settings

Check the universe, data paths, name cap, beta band, and both live-price age limits before making any files. The monthly workflow applies `LIVE_PRICE_REBALANCE_MAX_AGE_DAYS` immediately after its mandatory fresh WRDS pull; the separate panel staleness limit protects cached panel builds.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'config.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project folder containing config.py.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
import importlib
import config
import src.notebook_support as notebook_support

# Reload config first so a long-running kernel sees newly added settings.
config = importlib.reload(config)
notebook_support = importlib.reload(notebook_support)
settings_table = notebook_support.settings_table

display(settings_table())
print(f'Monthly workflow loaded from: {PROJECT_ROOT}')

## Step 2 — Refresh live prices from WRDS

This step always queries WRDS and intentionally has no cache toggle. It rebuilds `data/live_prices.parquet`, reports the exact latest price date and its calendar-day age, and carries every quarantine flag forward for review. If the newest price is more than seven calendar days old, the notebook raises and must not continue.

In [ ]:
from src.data import pull_live_prices
import importlib
import src.notebook_support as notebook_support

# Reload once more so this cell is safe even if the notebook was already open.
notebook_support = importlib.reload(notebook_support)

print('MONTHLY UPDATE DATA MODE: FRESH WRDS QUERY — LIVE-PRICE CACHE BYPASSED')
live_prices = pull_live_prices.main(refresh_from_wrds=True)
display(notebook_support.enforce_live_price_recency(live_prices))
display(notebook_support.confirm_live_prices(live_prices).tail(12))

## Step 3 — Rebuild and validate the panel

The panel builder appends only months after the CRSP cutoff, gives CRSP precedence in overlaps, refreshes Russell membership when a June live month is available, and applies the same investability rules. Zero blocking failures are required.

In [ ]:
from src.data import build_panel
from src.notebook_support import confirm_panel

panel, panel_checks = build_panel.main()
display(confirm_panel(panel, panel_checks))

## Step 4 — Generate the proposed target and trade list

This first prints the exact panel date that will be used as the signal date. It then scores that investable universe, applies the selection rule and portfolio constraints, and compares the proposal with `data/current_portfolio.csv`. A missing or empty current file raises unless `FIRST_REBALANCE = True` was explicitly set for the first launch. Existing dated target or trade files are never replaced unless the visible override below is changed by hand.

In [ ]:
from src.model.rebalance import run_rebalance
from src.notebook_support import announce_rebalance_signal_date, confirm_rebalance

rebalance_signal_date = announce_rebalance_signal_date(panel)
REPLACE_EXISTING_REBALANCE_FILES = False  # Change only after approving replacement.
rebalance = run_rebalance(
    rebalance_date=rebalance_signal_date,
    overwrite=REPLACE_EXISTING_REBALANCE_FILES,
)
display(confirm_rebalance(rebalance))

## Step 5 — MANUAL CHECKPOINT: review quarantined names

**Stop and read this table even when it is empty.** Every row is a liquid security whose provisional monthly return exceeded the quarantine threshold. Check the price history, split factor, exchange notice, and corporate action outside the model. A quarantined security is excluded from selection, but the flag still needs a human explanation.

- If the table is empty, record that the latest month had zero quarantines.
- If it has rows, assign an owner and document the conclusion for every name.
- Do not delete a row or overwrite a return in the notebook. Any repair must be reviewed and made in the source module.

In [ ]:
from src.notebook_support import quarantine_review

quarantined_names = quarantine_review(live_prices, latest_only=True)
display(quarantined_names)
print(f'MANUAL REVIEW REQUIRED: {len(quarantined_names)} quarantined rows in the latest live month.')

## Step 6 — MANUAL CHECKPOINT: review proposed trades

**These are proposals, not orders.** Review all warnings first, then every buy, sell, and material weight change. Confirm the reason, price and market-cap warnings, stale-fundamental warnings, sector weights, top positions, turnover, and beta. Resolve any `FAIL` before approving trades.

In [ ]:
from src.notebook_support import trade_review

review_tables = trade_review(rebalance)
for table_name, table in review_tables.items():
    print(table_name.upper())
    display(table)
print(f'MANUAL REVIEW REQUIRED: {len(rebalance.trades)} proposed trade rows; no trades have been executed.')

## Step 7 — Review what the proposed portfolio looks like today

This is the current-state view for discussion in the investment meeting. It uses the target already produced in Step 4 and shows concentration, active positions, sector and size positioning, beta, and the five block exposures. Historical performance remains in **04_backtest.ipynb** and is not repeated here.

In [ ]:
from src.model.portfolio_summary import run_portfolio_summary

current_portfolio_summary = run_portfolio_summary(
    rebalance=rebalance,
    panel=panel,
)
print(f'Current portfolio summary ready for {current_portfolio_summary.date:%Y-%m-%d}.')

## Step 8 — MANUAL CHECKPOINT: update current holdings only after execution

The notebook intentionally leaves `data/current_portfolio.csv` unchanged. After the trades are reviewed **and actually executed**, replace that file with the executed holdings using exactly two columns: `permno` and `weight`. Weights must describe the real portfolio, not the proposal.

Record who updated it and when. If trades were changed, partially filled, or not executed, use the actual resulting weights. Never copy the target file over current holdings before execution.

## What was produced and what comes next

This run refreshed `data/live_prices.parquet`, rebuilt the configured panel, wrote the date-stamped target and trade-list paths printed in Step 4, and printed the current target's holdings, active positions, sectors, size profile, and signal tilts. The quarantine review, warnings, and trade list require documented human sign-off.

After the target is approved, continue in the separate private trading repository; this model repository deliberately stops at the target and trade-list CSVs. After actual trading, update `data/current_portfolio.csv` by hand as described above. For a company-level discussion before approval, use **05_stock_lookup.ipynb**.

## If something fails

- **WRDS login or timeout:** verify the username in Step 1 and WRDS website access; retry once, then contact the WRDS administrator.
- **Missing panel or current-portfolio file:** run notebook 01 for the panel; obtain the agreed holdings file from the portfolio lead.
- **Live-price staleness error:** rerun Step 2. Do not relax the staleness setting to force a build.
- **Panel check failure:** stop. Do not generate or approve a rebalance from a failed panel.
- **Out-of-order cell:** restart the kernel and run from Step 1. See `START_HERE.ipynb` for detailed troubleshooting.